# Exact method: CP-SAT on three CVRPLIB instances

Upload A-n32-k5.vrp, A-n34-k5.vrp, and A-n37-k5.vrp when prompted. Execute all cells.

In [1]:
%pip -q install ortools numpy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 7.6 MB/s eta 0:00:00


In [7]:
from pathlib import Path
import math, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path("/content/cvrp_data") if Path("/content").exists() else Path("cvrp_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("/content/cvrp_outputs") if Path("/content").exists() else Path("cvrp_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# In Colab, upload the three actual .vrp files when prompted.
files_found = sorted(DATA_DIR.glob("*.vrp"))
if not files_found:
    try:
        from google.colab import files
        print("Upload your THREE CVRPLIB .vrp files (not screenshots).")
        uploaded = files.upload()
        for filename, data in uploaded.items():
            if filename.lower().endswith(".vrp"):
                (DATA_DIR / Path(filename).name).write_bytes(data)
    except ImportError:
        print(f"Local execution: copy the .vrp files into {DATA_DIR.resolve()}")
files_found = sorted(DATA_DIR.glob("*.vrp"))
if len(files_found) != 3:
    raise FileNotFoundError(f"Expected exactly 3 .vrp files in {DATA_DIR}; found {len(files_found)}: {[p.name for p in files_found]}")
print("Using actual files:", [p.name for p in files_found])

Using actual files: ['A-n32-k5.vrp', 'A-n34-k5.vrp', 'A-n37-k5.vrp']


In [8]:
def load_vrp(path):
    """Read standard CVRPLIB TSPLIB CVRP with NODE_COORD_SECTION, DEMAND_SECTION, DEPOT_SECTION."""
    text = Path(path).read_text(encoding="utf-8-sig")
    header, sections = {}, {}
    section = None
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line == "EOF": continue
        if line.endswith("_SECTION"):
            section = line
            sections[section] = []
        elif section:
            sections[section].append(line)
        else:
            if ":" in line:
                k, v = line.split(":", 1)
            else:
                tokens = line.split(None, 1)
                if len(tokens) != 2: continue
                k, v = tokens
            header[k.strip().upper()] = v.strip()
    if header.get("TYPE", "CVRP").upper() != "CVRP":
        raise ValueError(f"{path}: expected TYPE CVRP")
    if header.get("EDGE_WEIGHT_TYPE", "EUC_2D").upper() != "EUC_2D":
        raise ValueError(f"{path}: this notebook requires EUC_2D coordinates")
    coords_by_id = {}
    for row in sections.get("NODE_COORD_SECTION", []):
        vals = row.split()
        coords_by_id[int(vals[0])] = (float(vals[1]), float(vals[2]))
    demands_by_id = {}
    for row in sections.get("DEMAND_SECTION", []):
        vals = row.split()
        demands_by_id[int(vals[0])] = int(vals[1])
    depot_ids = [int(v.split()[0]) for v in sections.get("DEPOT_SECTION", [])
                 if int(v.split()[0]) != -1]
    if len(depot_ids) != 1:
        raise ValueError(f"{path}: expected exactly one depot, got {depot_ids}")
    depot = depot_ids[0]
    if set(coords_by_id) != set(demands_by_id):
        raise ValueError(f"{path}: coordinate and demand IDs differ")
    ids = [depot] + sorted(i for i in coords_by_id if i != depot)
    n = len(ids)
    if int(header.get("DIMENSION", n)) != n:
        raise ValueError(f"{path}: DIMENSION mismatch")
    coords = np.array([coords_by_id[i] for i in ids], dtype=float)
    demands = np.array([demands_by_id[i] for i in ids], dtype=int)
    if demands[0] != 0: raise ValueError("Depot demand must be zero")
    capacity = int(header["CAPACITY"])
    # CVRPLIB EUC_2D convention: nearest integer (nonnegative distances).
    dist = np.floor(np.linalg.norm(coords[:,None,:]-coords[None,:,:], axis=2)+0.5).astype(int)
    import re
    m = re.search(r"-k(\d+)", path.stem, re.I)
    if not m:
        raise ValueError(f"{path.name}: cannot infer fleet size from -kN filename; set it explicitly")
    vehicles = int(m.group(1))
    if np.any(demands[1:] > capacity): raise ValueError("Customer demand exceeds capacity")
    if demands.sum() > vehicles * capacity: raise ValueError("Fleet capacity is insufficient")
    return dict(name=path.stem, path=str(path), coords=coords, demands=demands,
                dist=dist, n=n, K=vehicles, capacity=capacity, ids=ids)

INSTANCES = [load_vrp(p) for p in files_found]
# Published best-known optima for these listed benchmark instances only.
KNOWN_OPTIMA = {"A-n32-k5":784, "A-n33-k5":661, "A-n34-k5":778,
                "A-n36-k5":799, "A-n37-k5":669}
info = pd.DataFrame([{"instance":i["name"], "customers":i["n"]-1,
                      "vehicles":i["K"], "capacity":i["capacity"],
                      "total_demand":int(i["demands"].sum()),
                      "published_optimum":KNOWN_OPTIMA.get(i["name"])}
                     for i in INSTANCES])
display(info)
info.to_csv(OUTPUT_DIR/"instance_summary.csv",index=False)

def route_cost(instance, route):
    d = instance["dist"]
    seq = [0] + list(route) + [0]
    return int(sum(d[a,b] for a,b in zip(seq, seq[1:])))

def solution_cost(instance, routes):
    return sum(route_cost(instance,r) for r in routes)

def validate(instance, routes):
    customers = [v for r in routes for v in r]
    expected = set(range(1,instance["n"]))
    counts = {v:customers.count(v) for v in customers}
    duplicate = sorted(v for v,c in counts.items() if c > 1)
    missing = sorted(expected-set(customers))
    unexpected = sorted(set(customers)-expected)
    loads = [int(sum(instance["demands"][v] for v in r if v in expected)) for r in routes]
    feasible = (not missing and not duplicate and not unexpected
                and len(routes) <= instance["K"]
                and all(0 < load <= instance["capacity"] for load in loads))
    return {"feasible":bool(feasible), "routes":len(routes), "loads":loads,
            "missing":missing,"duplicate":duplicate,"unexpected":unexpected}

def plot_routes(instance, routes, title, filename=None):
    xy = instance["coords"]
    fig, ax = plt.subplots(figsize=(7,6))
    ax.scatter(xy[1:,0],xy[1:,1],s=22,label="Customers",zorder=3)
    ax.scatter(xy[0,0],xy[0,1],marker="*",s=220,label="Depot",zorder=4)
    for j,r in enumerate(routes):
        seq = [0]+list(r)+[0]
        ax.plot(xy[seq,0],xy[seq,1],label=f"Vehicle {j+1}")
    ax.set(title=title,xlabel="X coordinate",ylabel="Y coordinate")
    ax.legend(fontsize=7,loc="best")
    fig.tight_layout()
    if filename: fig.savefig(OUTPUT_DIR/filename,dpi=160,bbox_inches="tight")
    plt.show()

,instance,customers,vehicles,capacity,total_demand,published_optimum
0,A-n32-k5,31,5,100,410,784
1,A-n34-k5,33,5,100,460,778
2,A-n37-k5,36,5,100,407,669


In [9]:
from ortools.sat.python import cp_model

def solve_exact(instance, time_limit_s=120, workers=8, seed=42):
    n, d, demands = instance["n"], instance["dist"], instance["demands"]
    K, Q = instance["K"], instance["capacity"]
    model = cp_model.CpModel()
    x = {(i,j):model.NewBoolVar(f"x_{i}_{j}")
         for i in range(n) for j in range(n) if i != j}
    model.AddMultipleCircuit([(i,j,var) for (i,j),var in x.items()])
    loads = {i:model.NewIntVar(int(demands[i]),Q,f"load_{i}") for i in range(1,n)}
    for i in range(1,n):
        for j in range(1,n):
            if i!=j:
                model.Add(loads[j]>=loads[i]+int(demands[j])).OnlyEnforceIf(x[i,j])
    model.Add(sum(x[0,j] for j in range(1,n)) <= K)
    model.Minimize(sum(int(d[i,j])*var for (i,j),var in x.items()))
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = float(time_limit_s)
    solver.parameters.num_search_workers = int(workers)
    solver.parameters.random_seed = int(seed)
    start = time.perf_counter()
    status = solver.Solve(model)
    runtime = time.perf_counter()-start
    result = {"instance":instance["name"],"method":"CP-SAT",
              "status":solver.StatusName(status),"runtime_s":runtime,
              "objective":None,"best_bound":None,"solver_gap_pct":None,
              "feasible":False,"routes":[]}
    if status in (cp_model.FEASIBLE,cp_model.OPTIMAL):
        outgoing = {}
        for (i,j),var in x.items():
            if solver.Value(var): outgoing.setdefault(i,[]).append(j)
        routes = []
        for first in outgoing.get(0,[]):
            route, cur, seen = [], first, set()
            while cur != 0:
                if cur in seen or cur not in outgoing or len(outgoing[cur])!=1:
                    raise RuntimeError("Could not extract valid solver routes")
                seen.add(cur)
                route.append(cur)
                cur=outgoing[cur][0]
            routes.append(route)
        validation = validate(instance,routes)
        if not validation["feasible"]:
            raise RuntimeError(f"Invalid CP-SAT solution: {validation}")
        obj = solution_cost(instance,routes)
        if obj != round(solver.ObjectiveValue()):
            raise RuntimeError("Extracted route cost disagrees with solver")
        bound = float(solver.BestObjectiveBound())
        result.update(objective=obj,best_bound=bound,
                      solver_gap_pct=100*(obj-bound)/obj if obj else 0,
                      feasible=True,routes=routes)
    return result

## Exact-method results (120-second limit per instance)
A solution with status FEASIBLE is not a solver proof of optimality.

In [ ]:
TIME_LIMIT_SECONDS=120
exact_rows=[]
for inst in INSTANCES:
    result=solve_exact(inst,time_limit_s=TIME_LIMIT_SECONDS)
    exact_rows.append({k:v for k,v in result.items() if k!='routes'})
    if result['feasible']:
        plot_routes(inst,result['routes'],f"{inst['name']} — CP-SAT",f"{inst['name']}_exact_routes.png")
exact_df=pd.DataFrame(exact_rows)
exact_df['published_optimum']=exact_df['instance'].map(KNOWN_OPTIMA)
exact_df['benchmark_gap_pct']=100*(exact_df['objective']-exact_df['published_optimum'])/exact_df['published_optimum']
display(exact_df)
exact_df.to_csv(OUTPUT_DIR/'exact_results.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4));ax.bar(exact_df.instance,exact_df.runtime_s);ax.set(ylabel='Seconds',title='CP-SAT runtime');fig.tight_layout();fig.savefig(OUTPUT_DIR/'exact_runtime.png',dpi=160);plt.show()
